# Mirage: the two external models on the fresh IMDb reviews (Kaggle, T4 GPU)

For the confirmatory check (G8, D53): Hugging Face's DistilBERT and RoBERTa-large on **10,000 IMDb
reviews never used before** (`imdb_fresh`) and their first + last sentences (`imdb_fresh_edges`).
Predictions only, no training, nothing to upload. Everything else in the check runs on the laptop.

1. Settings: **GPU T4 x1** and **Internet = On** (check Internet again before saving: it can switch
   off when a session restarts).
2. **Save Version → Save & Run All**. About 25–35 minutes. You can close the laptop.
3. When it's done, download `fresh_outputs.zip` from the output panel and unzip it into the repo
   root (`C:/claude/Mirage`). You should get 4 files in `results/predictions/`.

Every step stops the run with its real error message if it fails.

In [ ]:
REPO_URL = "https://github.com/AditeyaAItronics/mirage.git"

In [ ]:
import os, subprocess, sys


def run(*args):
    """Run a command, show its output as it goes, and stop the notebook if it fails."""
    print("$", " ".join(args), flush=True)
    p = subprocess.Popen(list(args), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise RuntimeError(f"step failed (exit code {p.returncode}): {' '.join(args)}")


if os.path.isdir("/kaggle/working"):
    os.chdir("/kaggle/working")
if os.path.basename(os.getcwd()) == "mirage":
    os.chdir("..")
if os.path.isdir("mirage"):  # throwaway clone: always move it to the latest commit
    run("git", "-C", "mirage", "fetch", "origin")
    run("git", "-C", "mirage", "reset", "--hard", "origin/main")
else:
    run("git", "clone", REPO_URL, "mirage")
os.chdir("mirage")
run("git", "log", "-1", "--oneline")
run(sys.executable, "-m", "pip", "install", "-q", "-e", ".[dl]")
run(sys.executable, "-c", "import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable a GPU!')")

In [ ]:
# The fresh sample is rebuilt from fixed seeds, exactly as on the laptop.
run(sys.executable, "-m", "mirage", "prepare")
run(sys.executable, "-c", "from mirage.data import prepare_fresh; print(prepare_fresh().to_string(index=False))")
import hashlib
import pandas as pd

fresh = pd.read_parquet("artifacts/data/imdb_fresh.parquet")
digest = hashlib.sha256("\n".join(fresh["text"]).encode()).hexdigest()[:16]
print(len(fresh), "reviews; first ids:", fresh["item_id"].head(3).tolist())
print("text fingerprint:", digest, "(must be 35935f818a427806)")
assert digest == "35935f818a427806", "the fresh reviews differ from the laptop's - stop and tell Claude"

In [ ]:
run(sys.executable, "-m", "mirage", "predict-external", "--datasets", "imdb_fresh", "imdb_fresh_edges")

In [ ]:
import glob, zipfile

out = "/kaggle/working/fresh_outputs.zip" if os.path.isdir("/kaggle/working") else "fresh_outputs.zip"
files = sorted(
    f
    for m in ("roberta_large", "distilbert_hf")
    for f in glob.glob(f"results/predictions/{m}__*__imdb_fresh*.parquet")
)
assert len(files) == 4, f"expected 4 prediction files, found {files}"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f)
print(out, os.path.getsize(out) // 2**10, "KB")
for f in files:
    p = pd.read_parquet(f)
    print(f, len(p), "rows, accuracy", round(p["correct"].mean(), 4))